# SARS — SDXL Reconstruction Pipeline
**Step 4: Citation-Grounded Image Generation**

Generates first-century BCE reconstructions of Templum Divi Augusti
using SDXL + ControlNet, grounded in VLM analysis and academic citations.

Run with T4 or A100 GPU runtime. Requires VLM analysis outputs to exist in Drive.

In [ ]:
# ── CELL 1: HARDWARE CHECK ────────────────────────────────────────────────
import torch

print('=' * 50)
print('HARDWARE CHECK')
print('=' * 50)

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✓ GPU: {gpu}')
    print(f'✓ VRAM: {vram:.1f} GB')
    if 'T4' in gpu:
        print('✓ T4 — SDXL: ~3-5 min/image')
    elif 'A100' in gpu:
        print('✓ A100 — SDXL: ~30-45 sec/image')
    else:
        print('  GPU detected — estimating ~2-5 min/image')
else:
    print('✗ NO GPU DETECTED')
    print('  Go to: Runtime → Change runtime type → T4 GPU')
    raise RuntimeError('GPU required for SDXL. Please enable T4 GPU runtime.')

print('=' * 50)

In [ ]:
# ── CELL 2: INSTALL DEPENDENCIES ──────────────────────────────────────────
print('Installing dependencies...')

%pip install -q -U diffusers transformers accelerate huggingface_hub
%pip install -q langchain langchain-community langchain-huggingface
%pip install -q langchain-chroma chromadb sentence-transformers
%pip install -q xformers psutil

print('✓ Done')

In [ ]:
# ── CELL 3: PROJECT SETUP ─────────────────────────────────────────────────
import sys
import os
from google.colab import drive

drive.mount('/content/drive')

_drive_root = '/content/drive/MyDrive'
_candidates = ['SARS', 'SARS_colab', 'sars']
PROJECT = None
for _name in _candidates:
    _path = os.path.join(_drive_root, _name)
    if os.path.isdir(_path):
        PROJECT = _path
        break

if PROJECT is None:
    _found = [d for d in os.listdir(_drive_root)
              if os.path.isdir(os.path.join(_drive_root, d))]
    raise FileNotFoundError(
        f'Could not find SARS project folder in Google Drive.\n'
        f'Looked for: {_candidates}\n'
        f'Folders found in MyDrive: {_found}\n'
        f'Rename your Drive folder to "SARS" or update _candidates above.'
    )

os.chdir(PROJECT)
sys.path.insert(0, PROJECT)

REQUIRED_PATHS = [
    'data/chroma_db',
    'data/conditioning/registry/conditioning_registry.json',
    'data/analysis/vlm_outputs',
    'src/generation.py',
]

print(f'Project: {PROJECT}')
all_ok = True
for p in REQUIRED_PATHS:
    exists = os.path.exists(p)
    print(f"  {'✓' if exists else '✗ MISSING'}  {p}")
    if not exists:
        all_ok = False

if not all_ok:
    raise FileNotFoundError('Some required paths are missing. Check your Drive upload.')

print('\n✓ Project ready')

In [ ]:
# ── CELL 4: REMAP REGISTRY PATHS ──────────────────────────────────────────
import json
from pathlib import Path

LOCAL_PREFIX = '/home/ece/VSCodeProjects/SARS'
COLAB_PREFIX = PROJECT

registry_path = 'data/conditioning/registry/conditioning_registry.json'
with open(registry_path, encoding='utf-8') as f:
    registry = json.load(f)

for entry in registry:
    for key in ('source_path', 'canny_path', 'depth_path'):
        if key in entry and entry[key]:
            entry[key] = entry[key].replace(LOCAL_PREFIX, COLAB_PREFIX)

with open(registry_path, 'w', encoding='utf-8') as f:
    json.dump(registry, f, indent=2)

print(f'✓ Registry paths remapped to {COLAB_PREFIX}')

In [ ]:
# ── CELL 5: LOAD SHARED RESOURCES ─────────────────────────────────────────
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

CHROMA_DIR = 'data/chroma_db'
CONDITIONING_REGISTRY = 'data/conditioning/registry/conditioning_registry.json'
ANALYSIS_DIR = 'data/analysis/vlm_outputs'
OUTPUT_DIR = 'data/outputs'

print('Loading vector store...')
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True},
)
vs = Chroma(
    collection_name='augustus_temple',
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR,
)
print(f'✓ Vector store: {vs._collection.count()} chunks')

In [ ]:
# ── CELL 6: CHECK VLM ANALYSIS OUTPUTS ────────────────────────────────────
from pathlib import Path

outputs = list(Path(ANALYSIS_DIR).glob('*_analysis.json'))
completed = [f for f in outputs if '"error"' not in f.read_text(encoding='utf-8')]
failed = [f for f in outputs if '"error"' in f.read_text(encoding='utf-8')]
with_prompt = [
    f for f in completed
    if '"sdxl_prompt_component"' in f.read_text(encoding='utf-8')
]

print(f'Total analyzed      : {len(outputs)}')
print(f'Completed           : {len(completed)}')
print(f'With SDXL prompt    : {len(with_prompt)}')
print(f'Failed/errors       : {len(failed)}')

if len(with_prompt) == 0:
    raise RuntimeError(
        'No completed VLM analyses found. '
        'Run colab_pipeline.ipynb Cell 5 (VLM batch) first.'
    )

In [ ]:
# ── CELL 7: LOAD SDXL PIPELINE ────────────────────────────────────────────
# Downloads ~7GB on first run, then cached.
# Inpaint pipe shares weights with the base pipe (no extra VRAM, no
# extra download) — used by stage 2 to complete blank/unfinished regions.
from src.generation import detect_hardware, load_sdxl_pipeline, load_sdxl_inpaint_pipeline

hw = detect_hardware()
print('\nLoading SDXL + Canny ControlNet...')
pipe = load_sdxl_pipeline(hw, use_canny=True, use_depth=False)
print('Loading SDXL inpaint stage (shared weights)...')
inpaint_pipe = load_sdxl_inpaint_pipeline(hw, base_pipe=pipe)
print('✓ Pipeline ready (stage 1 ControlNet + stage 2 inpaint)')

In [ ]:
# ── CELL 8: TEST GENERATION (1 image, 20 steps) ───────────────────────────
import json
import time
from pathlib import Path
from src.generation import _score_analysis, generate_reconstruction

analyses = []
for f in Path(ANALYSIS_DIR).glob('*_analysis.json'):
    data = json.loads(f.read_text(encoding='utf-8'))
    if 'error' not in data and data.get('sdxl_prompt_component'):
        analyses.append(data)

best = max(analyses, key=_score_analysis)
print(f'Test candidate : {best["source_filename"]}')
print(f'Score          : {_score_analysis(best)}')
print(f'Credibility    : {best["credibility_tier"]}')
print(f'Mosque         : {best.get("mosque_interference", "N/A")}')
print()

start = time.time()
result = generate_reconstruction(
    analysis=best,
    conditioning_registry_path=CONDITIONING_REGISTRY,
    pipe=pipe,
    hw=hw,
    vector_store=vs,
    output_dir=OUTPUT_DIR,
    num_inference_steps=20,
    guidance_scale=11.0,
    use_canny=True,
    use_depth=False,
)
elapsed = time.time() - start
print(f'\n✓ Done in {elapsed/60:.1f} minutes')
print(f'  Reconstruction : {result["reconstruction_path"]}')
print(f'  Provenance     : {result["provenance_path"]}')

In [ ]:
# ── CELL 8b: PREVIEW AGGREGATED PROMPT ────────────────────────────────────
from src.generation import (
    aggregate_all_analyses,
    build_aggregated_prompt,
    load_plan_context,
)

plan_ctx = load_plan_context(ANALYSIS_DIR)
print(f"Plan context available: {'yes' if plan_ctx else 'no'}")
if plan_ctx:
    print(f"  {plan_ctx[:100]}...")

print("\nAggregating all analyses...")
aggregated = aggregate_all_analyses(ANALYSIS_DIR)

print(f"\nSources aggregated : {aggregated['source_count']}")
print(f"Confirmed elements : {len(aggregated['confirmed_elements'])}")
print(f"Unique citations   : {len(aggregated['all_citations'])}")
print(f"Mosque dominance   : {aggregated['dominant_mosque_interference']}")

print(f"\nTop confirmed elements:")
for e in aggregated['confirmed_elements'][:10]:
    print(f"  • {e}")

positive, negative = build_aggregated_prompt(aggregated, vs)

print(f"\nPositive prompt ({len(positive)} chars / {len(positive.split())} words):")
print(positive[:300] + "...")

print("\nReady to generate 4 canonical views.")
print("Each takes ~4-5 min on T4, ~30-45 sec on A100.")
print("\nRun Cell 8c to start generation.")

In [ ]:
# ── CELL 8c: GENERATE CANONICAL VIEWS ────────────────────────────────────
# Requires Cell 8b to have been run first.
import time
from src.generation import generate_canonical_views

start_all = time.time()

canonical_results = generate_canonical_views(
    analysis_dir=ANALYSIS_DIR,
    conditioning_registry_path=CONDITIONING_REGISTRY,
    pipe=pipe,
    hw=hw,
    vector_store=vs,
    output_dir=OUTPUT_DIR,
    num_inference_steps=40,
    guidance_scale=11.0,
    plan_context=plan_ctx,
    inpaint_pipe=inpaint_pipe,
)

total_elapsed = time.time() - start_all
print(f"\n✓ All canonical views complete")
print(f"  Total time: {total_elapsed/60:.1f} minutes")

In [ ]:
# ── CELL 8d: DISPLAY CANONICAL VIEWS ─────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig, axes = plt.subplots(2, 2, figsize=(20, 20))
axes = axes.flatten()

for ax, result in zip(axes, canonical_results):
    img = mpimg.imread(result['reconstruction_path'])
    ax.imshow(img)
    ax.set_title(result['title'], fontsize=14, fontweight='bold')
    ax.axis('off')

for ax in axes[len(canonical_results):]:
    ax.axis('off')

n_confirmed = len(aggregated['confirmed_elements'])
plt.suptitle(
    f"Templum Divi Augusti — Ankara, 25 BCE\n"
    f"Citation-Grounded Reconstruction\n"
    f"Aggregated from {n_confirmed} confirmed elements across "
    f"{aggregated['source_count']} analyses",
    fontsize=14,
    fontweight='bold',
)
plt.tight_layout()
plt.show()

print("\nALL CITATIONS USED:")
print(f"(aggregated from {aggregated['source_count']} analyses)")
for c in aggregated['all_citations']:
    print(f"  • {c}")

In [ ]:
# ── CELL 9: DISPLAY TEST RESULT ───────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

recon = mpimg.imread(result['reconstruction_path'])
axes[0].imshow(recon)
axes[0].set_title(
    'First Century Reconstruction\n(Citation-Grounded)',
    fontsize=12, fontweight='bold'
)
axes[0].axis('off')

prov = result['provenance']
citations = prov.get('citations_used') or prov.get('all_citations', [])
summary = (
    f"Source: {prov.get('source_image', 'N/A')}\n"
    f"Credibility: {prov.get('credibility_tier', 'aggregated')}\n"
    f"Mosque: {prov.get('mosque_interference', 'N/A')}\n\n"
    f"Citations:\n" +
    '\n'.join(f"• {c}" for c in citations[:5])
)
axes[1].text(
    0.05, 0.95, summary,
    transform=axes[1].transAxes,
    fontsize=9, verticalalignment='top',
    fontfamily='monospace',
    bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8)
)
axes[1].axis('off')
axes[1].set_title('Provenance', fontsize=12)

plt.suptitle(
    'Templum Divi Augusti — Ankara, 25 BCE',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.show()

In [ ]:
# ── CELL 10: FULL BATCH GENERATION ────────────────────────────────────────
# Generates reconstructions for all analyzed images.
# resume=True skips already-generated outputs.
# Set max_images=5 for a test run.
from src.generation import run_batch_generation

results_gen = run_batch_generation(
    analysis_dir=ANALYSIS_DIR,
    conditioning_registry_path=CONDITIONING_REGISTRY,
    vector_store=vs,
    output_dir=OUTPUT_DIR,
    pipe=pipe,
    hw=hw,
    num_inference_steps=40,
    guidance_scale=11.0,
    resume=True,
    max_images=None,  # set to 5 for testing
    inpaint_pipe=inpaint_pipe,
)

In [ ]:
# ── CELL 11: RESULTS GRID ─────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

recon_dir = Path(OUTPUT_DIR) / 'reconstructions'
images = sorted(recon_dir.glob('*.png'))
print(f'Total reconstructions: {len(images)}')

cols = 3
rows = max(1, (len(images) + cols - 1) // cols)
fig, axes = plt.subplots(rows, cols, figsize=(18, 6 * rows))
axes_flat = axes.flatten() if rows > 1 else (list(axes) if cols > 1 else [axes])

for ax, img_path in zip(axes_flat, images):
    ax.imshow(mpimg.imread(str(img_path)))
    ax.set_title(img_path.stem[:40], fontsize=7)
    ax.axis('off')

for ax in axes_flat[len(images):]:
    ax.axis('off')

plt.suptitle(
    'All Reconstructions — Templum Divi Augusti',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.show()